<a href="https://colab.research.google.com/github/omarsamehabobaker619-bot/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarsamehabobaker619-bot/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane:** CTR / Engagement Opportunity Scoring

I'm picking this lane because the columns it needs (impressions, average position,
CTR) are ones I already touched in Notebook 02, so I understand what they represent
before I start modeling. The core idea is also easy to state precisely: some pages
get plenty of visibility (high impressions, decent position) but still don't get
clicked much — that's a fixable content/metadata problem, distinct from a page that
simply isn't being shown at all. That specificity makes it easier to turn into a
concrete recommendation later, rather than a vague "improve rankings" goal.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Unit of analysis:** a single page (one row = one page's aggregated search stats).

**The question:** Among pages with meaningful impressions, which ones are
under-capturing clicks relative to their position — i.e., have CTR noticeably
lower than similar pages at a similar rank?

**The decision this supports:** whether a page's title/meta description/snippet
is worth rewriting versus leaving alone.

**Who acts on it:** a content editor or SEO specialist deciding where to spend
limited rewrite time this week.

**Cost of a wrong call:**
- False positive (flagged as underperforming, but it's actually fine): wasted
  editor time on a rewrite that wasn't needed.
- False negative (a genuinely underperforming page never gets flagged): a page
  keeps losing clicks it could otherwise capture, quietly, with no one looking at it.

**Why data/ML helps here:** with thousands of pages, no one can eyeball every
impressions-vs-CTR pair by hand — a simple, readable model can rank pages by how
unusual their CTR is for their position, so a person only has to review a short list.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [4]:
import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Clean up infinities the same way earlier notebooks did
df_clean = df.replace([np.inf, -np.inf], np.nan)

# Number 1: how many pages have real visibility (impressions) to begin with
visible = df_clean[df_clean["impressions_90d"] > 0]
print("Pages with impressions in the last 90 days:", len(visible), "out of", len(df_clean))

# Number 2: among well-positioned pages, how spread out is CTR?
good_position = visible[visible["avg_position"] <= visible["avg_position"].median()]
print("Median CTR among good-position pages:", good_position["ctr"].median())
print("25th percentile CTR among good-position pages:", good_position["ctr"].quantile(0.25))

# Number 3: how many pages sit in the "good position, low CTR" zone —
# i.e. plausible candidates for this lane
low_ctr_cutoff = good_position["ctr"].quantile(0.25)
candidates = good_position[good_position["ctr"] <= low_ctr_cutoff]
print("Pages with good position but bottom-quartile CTR:", len(candidates))
print("As % of all visible pages:", round(100 * len(candidates) / len(visible), 1), "%")

Pages with impressions in the last 90 days: 30000 out of 30000
Median CTR among good-position pages: 0.11
25th percentile CTR among good-position pages: 0.0
Pages with good position but bottom-quartile CTR: 6162
As % of all visible pages: 20.5 %


**What this shows:**

All 30,000 pages in the starter dataset have real impressions in the last 90 days —
so visibility isn't the bottleneck for this sample; it's what happens after a page
is shown.

Among pages with good average position, median CTR is 0.11 — but the 25th percentile
is exactly 0.0. That means at least a quarter of well-positioned pages get *no clicks
at all* despite being shown often and ranked reasonably well. That's a real,
non-trivial gap between visibility and engagement.

Using a simple bottom-quartile cutoff, 6,162 pages (20.5% of all visible pages) fall
into this "good position, low CTR" zone — a large enough pool that a systematic
scoring approach is worth building, rather than reviewing pages one by one.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**

- **Observed patterns** in this specific starter dataset (30,000 anonymized pages) —
  for example, that a meaningful share of well-positioned pages still get very low
  or zero CTR.
- **Directional signals**: which page characteristics are *associated with* low CTR,
  based on correlation and simple readable models — not a guarantee that changing
  those characteristics will fix the problem.
- **Decision support**: a ranked shortlist of pages worth a human's attention, with
  reasons attached — not an automatic fix, and not a guarantee of improved clicks
  if acted on.

**What I can never claim:**

- **Causation.** A page having low CTR alongside certain features does not mean
  those features *caused* the low CTR. Other unmeasured factors (search intent,
  competitor snippets, seasonality) could easily explain the same pattern.
- **Predicting or reverse-engineering Google's ranking algorithm.** This work only
  studies observable outcomes (position, impressions, clicks) in an anonymized
  sample — it says nothing about how Google's systems actually work internally.
- **Guaranteed results from acting on recommendations.** Rewriting a flagged page's
  title or meta description may or may not raise its CTR — this analysis identifies
  *candidates worth reviewing*, not proven fixes.
- **Generalization beyond this sample.** Findings here reflect this specific
  anonymized dataset and time window, not all websites or all search queries
  universally.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] Every section filled — markdown thinking and supporting code
- [x] Notebook runs top to bottom with no errors (Restart session and Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Committed to work/notebooks/ in my repo